In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.0 MB/s eta 0:00:00


In [ ]:
import os
import pickle
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA="/content/drive/MyDrive/FIFI_Research/data"

train_df=pd.read_csv(
    os.path.join(DATA,"train.tsv"),
    sep="\t"
)

val_df=pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

print(train_df.shape)
print(val_df.shape)

(90000, 4)
(18000, 4)


In [ ]:
MODEL_PATH="/content/drive/MyDrive/FIFI_Research/models"

with open(
    os.path.join(MODEL_PATH,"candidate_titles.pkl"),
    "rb"
) as f:
    candidate_titles=pickle.load(f)

candidate_embeddings=np.load(
    os.path.join(MODEL_PATH,"candidate_embeddings.npy")
)

index=faiss.read_index(
    os.path.join(MODEL_PATH,"faiss.index")
)

model=SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from collections import Counter
import re

word_counter=Counter()

for title in train_df["original_title"]:

    words=re.findall(r"[A-Za-z]+",title.lower())

    word_counter.update(words)

In [ ]:
domain_words=[]

for word,count in word_counter.items():

    if count>50 and len(word)>4:

        domain_words.append(word)

print(len(domain_words))

1706


In [ ]:
expansion_rules={

"image":[
"vision",
"segmentation",
"detection",
"classification",
"cnn"
],

"learning":[
"deep",
"neural",
"representation",
"training",
"model"
],

"network":[
"graph",
"neural",
"architecture",
"learning"
],

"brain":[
"fmri",
"eeg",
"neural",
"cognitive"
],

"robot":[
"robotics",
"navigation",
"planning",
"control"
],

"language":[
"nlp",
"bert",
"transformer",
"text"
],

"video":[
"tracking",
"motion",
"recognition"
],

"detection":[
"recognition",
"classification",
"segmentation"
],

"classification":[
"prediction",
"recognition",
"learning"
]
}

In [ ]:
def expand_query(query):

    query_lower=query.lower()

    expansion=[]

    for key in expansion_rules:

        if key in query_lower:

            expansion.extend(
                expansion_rules[key]
            )

    expansion=list(set(expansion))

    expanded=query+" "+" ".join(expansion)

    return expanded

In [ ]:
val_df["expanded_query"]=val_df["generated_title"].apply(
    expand_query
)

val_df[
    [
        "generated_title",
        "expanded_query"
    ]
].head()

,generated_title,expanded_query
0,Fully Convolutional Joint Detection and Regres...,Fully Convolutional Joint Detection and Regres...
1,When Machines Argue: Teaching AI to Spot Fake ...,When Machines Argue: Teaching AI to Spot Fake ...
2,Optimized 2D Manifold Folding and Attribute Ma...,Optimized 2D Manifold Folding and Attribute Ma...
3,From Snapshot to Sawdust: Rebuilding Wooden Ob...,From Snapshot to Sawdust: Rebuilding Wooden Ob...
4,"A Survey of How Deep Learning Improves Image, ...","A Survey of How Deep Learning Improves Image, ..."


In [ ]:
expanded_embeddings=model.encode(

    val_df["expanded_query"].tolist(),

    normalize_embeddings=True,

    batch_size=64,

    show_progress_bar=True
)

Batches:   0%|          | 0/282 [00:00<?, ?it/s]

In [ ]:
faiss_scores, faiss_indices = index.search(
    expanded_embeddings,
    10
)

In [ ]:
retrieved=[]

for row in faiss_indices:

    retrieved.append(

        [

            candidate_titles[i]

            for i in row

        ]

    )

In [ ]:
def reciprocal_rank(predictions,gt):

    for rank,title in enumerate(predictions,1):

        if title==gt:

            return 1/rank

    return 0

rr=[]

for idx in range(len(val_df)):

    rr.append(

        reciprocal_rank(

            retrieved[idx],

            val_df.iloc[idx]["original_title"]

        )

    )

mrr=np.mean(rr)

print("MRR@10 =",round(mrr,4))

MRR@10 = 0.7882


In [ ]:
results=[]

for style in [

"technical",

"accessible",

"catchy"

]:

    subset = val_df[
        val_df["category"] == style
    ]

    style_scores = []

    for idx in subset.index:

        style_scores.append(

            reciprocal_rank(

                retrieved[idx],

                val_df.loc[idx,"original_title"]

            )

        )

    results.append({

        "Style": style,

        "MRR@10": np.mean(style_scores)

    })

results = pd.DataFrame(results)

results

,Style,MRR@10
0,technical,0.940490
1,accessible,0.670649
2,catchy,0.753490


In [ ]:
submission = []

for i in range(len(val_df)):

    query_id = val_df.iloc[i]["id"]

    for rank in range(10):

        submission.append({

            "id": query_id,

            "rank": rank + 1,

            "score": float(faiss_scores[i][rank]),

            "original_title": candidate_titles[
                faiss_indices[i][rank]
            ]

        })

submission = pd.DataFrame(submission)

print("Submission Shape:", submission.shape)

submission.head(20)

Submission Shape: (180000, 4)


,id,rank,score,original_title
0,0,1,0.916213,Preterm infants' limb-pose estimation from dep...
1,0,2,0.739023,Collaborative Descriptors: Convolutional Maps ...
2,0,3,0.737754,Training Bit Fully Convolutional Network for F...
3,0,4,0.735901,TOOD: Task-aligned One-stage Object Detection
4,0,5,0.734120,CompConv: A Compact Convolution Module for Eff...
5,0,6,0.733673,"BWCNN: Blink to Word, a Real-Time Convolutiona..."
6,0,7,0.731898,Back to the Future: Joint Aware Temporal Deep ...
7,0,8,0.730871,Speed/accuracy trade-offs for modern convoluti...
8,0,9,0.728096,Object Specific Deep Learning Feature and Its ...
9,0,10,0.727908,Iterative Multi-domain Regularized Deep Learni...


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/FIFI_Research/submissions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

submission.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "FutureMinds_task1_run2.tsv"

    ),

    sep="\t",

    index=False

)

print("Saved Successfully!")

print(submission.head(20))

Saved Successfully!
    id  rank     score                                     original_title
0    0     1  0.916213  Preterm infants' limb-pose estimation from dep...
1    0     2  0.739023  Collaborative Descriptors: Convolutional Maps ...
2    0     3  0.737754  Training Bit Fully Convolutional Network for F...
3    0     4  0.735901      TOOD: Task-aligned One-stage Object Detection
4    0     5  0.734120  CompConv: A Compact Convolution Module for Eff...
5    0     6  0.733673  BWCNN: Blink to Word, a Real-Time Convolutiona...
6    0     7  0.731898  Back to the Future: Joint Aware Temporal Deep ...
7    0     8  0.730871  Speed/accuracy trade-offs for modern convoluti...
8    0     9  0.728096  Object Specific Deep Learning Feature and Its ...
9    0    10  0.727908  Iterative Multi-domain Regularized Deep Learni...
10   1     1  0.753897  Towards Automated Factchecking: Developing an ...
11   1     2  0.751652  Detecting Deception in Political Debates Using...
12   1     3  0.74

In [ ]:
np.save(
    os.path.join(MODEL_PATH, "run2_faiss_scores.npy"),
    faiss_scores
)

np.save(
    os.path.join(MODEL_PATH, "run2_faiss_indices.npy"),
    faiss_indices
)

print("Run2 retrieval results saved successfully.")

Run2 retrieval results saved successfully.
